# **Implementing the Collaborative Filtering Algorithm from Scratch**

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
X = np.loadtxt('small_movies_X.csv', delimiter=',')
W = np.loadtxt('small_movies_W.csv', delimiter=',')
b = np.loadtxt('small_movies_b.csv', delimiter=',')

Y = np.loadtxt('small_movies_Y.csv', delimiter=',')
R = np.loadtxt('small_movies_R.csv', delimiter=',')

In [3]:
print(X.shape) # i'th row is feature vector for the i-th movie
print(W.shape) # j'th row is parameter vector for the j-th user
print(b.shape)

print(Y.shape) # matrix that stores user ratings of movies
print(R.shape) # matrix that stores whether a user has rated a movie or not (1 or 0)

(4778, 10)
(443, 10)
(443,)
(4778, 443)
(4778, 443)


In [4]:
print('Number of movies: ', Y.shape[0])
print('Number of users: ', Y.shape[1])
print('Number of features: ', X.shape[1])

Number of movies:  4778
Number of users:  443
Number of features:  10


In [5]:
print('Average rating for movie 0: ', np.mean(Y[0, R[0, :] == 1]))

Average rating for movie 0:  3.4


In [6]:
def cost_function(X, W, b, Y, R, lambda_): #mean squared error cost function
    num_users = Y.shape[1]
    num_movies = Y.shape[0]
    J = 0
    for j in range(num_users):
        w = W[j, :]
        b_j = b[0, j]
        for i in range(num_movies):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += r * (np.dot(w, x) + b_j - y) ** 2
    J += (lambda_) * (np.sum(W ** 2) + np.sum(X ** 2))
    J = J / 2
    
    return J

In [7]:
def vec_cost_function(X, W, b, Y, R, lambda_): #vectorized mean squared error cost function
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J 

In [8]:
my_ratings = np.zeros(Y.shape[0]) # Y.shape[0] is number of movies

my_ratings[2700] = 5   # Toy Story 3 (2010)
my_ratings[2609] = 2  # Persuasion (2007)
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003) 

In [9]:
my_ratings.shape

(4778,)

In [10]:
def normalise_rating(Y, R):
    Ymean = (np.sum(Y*R, axis=1) / (np.sum(R, axis=1) + 1e-12)).reshape(-1, 1)
    Ynorm = Y - np.multiply(Ymean, R)
    return (Ynorm, Ymean)

In [11]:
Y = np.c_[my_ratings, Y]  # Add my ratings as the first column of Y
R = np.c_[(my_ratings != 0).astype(int), R]  # Add my mask as the first column of R

In [12]:
print('New shape of Y:', Y.shape)
print('New shape of R:', R.shape)

New shape of Y: (4778, 444)
New shape of R: (4778, 444)


In [13]:
Ynorm, Ymean = normalise_rating(Y, R)

In [14]:
num_movies, num_users = Y.shape
num_features = 100

tf.random.set_seed(1234)
# Initializing the parameters W, X, b using tf.Variable
W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64), name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features), dtype=tf.float64), name='X')
b = tf.Variable(tf.random.normal((1, num_users), dtype=tf.float64), name='b')

optimizer = keras.optimizers.Adam(learning_rate=1e-1)

In [15]:
# Custmom training loop
iterations = 200
lambda_ = 1

for i in range(iterations):
    with tf.GradientTape() as tape:
        cost = vec_cost_function(X,W, b, Ynorm, R, lambda_) # computing cost
    gradients = tape.gradient(cost, [X, W, b]) # using the gradients from gradient tape with respect to the parameters and the loss
    optimizer.apply_gradients(zip(gradients, [X, W, b])) # running one step of gradient descent to minimize the cost function

    if i % 20 == 0:
        print(f"Training loss at iteration {i}: {cost:0.1f}")

Training loss at iteration 0: 2321191.3
Training loss at iteration 20: 136169.3
Training loss at iteration 40: 51863.7
Training loss at iteration 60: 24599.0
Training loss at iteration 80: 13630.6
Training loss at iteration 100: 8487.7
Training loss at iteration 120: 5807.8
Training loss at iteration 140: 4311.6
Training loss at iteration 160: 3435.3
Training loss at iteration 180: 2902.1


In [16]:
predictions = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()  
predictions += Ymean # restoring the mean
my_predictions = predictions[:,0]  

In [17]:
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Actual rating for movie {i}: {my_ratings[i]:0.1f}')
        print(f'Predicted rating for movie {i}: {my_predictions[i]:0.1f}')
        print()

Actual rating for movie 246: 5.0
Predicted rating for movie 246: 4.9

Actual rating for movie 366: 5.0
Predicted rating for movie 366: 4.8

Actual rating for movie 382: 2.0
Predicted rating for movie 382: 2.1

Actual rating for movie 622: 5.0
Predicted rating for movie 622: 4.9

Actual rating for movie 793: 5.0
Predicted rating for movie 793: 4.9

Actual rating for movie 929: 5.0
Predicted rating for movie 929: 4.9

Actual rating for movie 988: 3.0
Predicted rating for movie 988: 3.0

Actual rating for movie 1150: 5.0
Predicted rating for movie 1150: 4.9

Actual rating for movie 2609: 2.0
Predicted rating for movie 2609: 2.1

Actual rating for movie 2700: 5.0
Predicted rating for movie 2700: 4.8

Actual rating for movie 2716: 3.0
Predicted rating for movie 2716: 3.0

Actual rating for movie 2925: 1.0
Predicted rating for movie 2925: 1.4

Actual rating for movie 2937: 1.0
Predicted rating for movie 2937: 1.3

